In [5]:
import json
import pandas as pd
from datasets import Dataset

DATA_PATH = r"data\processed\final_dataset\final_dataset.jsonl"

records = []

relative_path = Path(DATA_PATH.replace("\\", "/"))

if relative_path.is_absolute():
    data_file = relative_path
else:
    candidates = [base / relative_path for base in [Path.cwd(), *Path.cwd().parents]]
    data_file = next((path for path in candidates if path.is_file()), None)

if data_file is None:
    raise FileNotFoundError(
        f"Could not find {DATA_PATH!r}. Current working directory: {Path.cwd()}"
    )

with data_file.open("r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

print(f"Loaded {len(records)} notices total")


formatted_rows = []
skipped_no_text = 0
for item in records:
    raw_notice = (item.get("cleaned_text") or item.get("raw_text") or "").strip()
    if not raw_notice:
        skipped_no_text += 1
        continue

    student_summary = item.get("student_summary")
    if student_summary:
        formatted_rows.append({
            "input_text": f"summarize for student: {raw_notice}",
            "target_summary": student_summary,
        })

    faculty_summary = item.get("faculty_summary")
    if faculty_summary:
        formatted_rows.append({
            "input_text": f"summarize for faculty: {raw_notice}",
            "target_summary": faculty_summary,
        })

print(f"Skipped {skipped_no_text} notices with no extracted text")
print(f"Built {len(formatted_rows)} (input, target) training rows from labeled summaries")

if len(formatted_rows) < 20:
    print(
        "WARNING: this is a very small number of labeled examples for "
        "fine-tuning a seq2seq model. Expect the model to memorize/overfit "
        "rather than generalize -- see the note at the end of the notebook."
    )

raw_dataset_full = Dataset.from_pandas(pd.DataFrame(formatted_rows))
split = raw_dataset_full.train_test_split(test_size=0.2, seed=42)
raw_dataset = split["train"]
eval_dataset = split["test"]

print("\nSample Dataset Row:")
print(raw_dataset[0])
print(f"\nTrain rows: {len(raw_dataset)} | Eval rows: {len(eval_dataset)}")

Loaded 75 notices total
Skipped 0 notices with no extracted text
Built 23 (input, target) training rows from labeled summaries

Sample Dataset Row:
{'input_text': 'summarize for student: Annexure-7.1\n(Enclosure to Item No.7)\n\nIndian Institute of Technology Bombay\nOffice of the Dean (Student Affairs)\nCampus Code of Conduct\n\n1. Basic policy governing student life\n a) Every student has the right to all the advantages, prestige and honours accruing to a student of this Institute.\n b) The Institute will endeavour to provide a living and learning environment in which the student can meet her/his academic goals.\nThe Institute has the responsibility of providing the student with a clear understanding of its academic requirements which are generally set forth in writing in the Institute’s brochures, rules, and regulations.\n c) The Institute will determine when its rules are violated and to determine the appropriate course of action. By enrolling in the\nInstitute, the student accepts

In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer

model_name = "facebook/bart-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

def preprocess_function(examples):
    model_inputs = tokenizer(examples["input_text"], max_length=512, truncation=True, padding="max_length")
    labels = tokenizer(text_target=examples["target_summary"], max_length=128, truncation=True, padding="max_length")
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_dataset = raw_dataset.map(preprocess_function, batched=True)
tokenized_eval_dataset = eval_dataset.map(preprocess_function, batched=True)

training_args = Seq2SeqTrainingArguments(
    output_dir="./results_bart",
    num_train_epochs=10,             
    per_device_train_batch_size=2,
    eval_strategy="epoch",
    logging_steps=1,
    save_strategy="no",
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    eval_dataset=tokenized_eval_dataset,
    processing_class=tokenizer,
)

print("Starting Fine-Tuning Loop...")
trainer.train()
print("Training Complete!")

Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

Map:   0%|          | 0/18 [00:00<?, ? examples/s]

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

Starting Fine-Tuning Loop...


c:\ProjectFiles\uni-comms-intelligence\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,6.671217,3.932770
2,5.562098,3.144864
3,4.294781,2.594083
4,3.954621,2.292668
5,2.393832,2.076381
6,2.350483,1.967690
7,2.210067,1.844021
8,1.350317,1.773193
9,2.291584,1.732626
10,1.392114,1.719086


Training Complete!


In [ ]:
def generate_persona_summary(raw_notice: str, role: str = "student") -> str:
    input_prompt = f"summarize for {role}: {raw_notice}"
    inputs = tokenizer(input_prompt, return_tensors="pt", truncation=True, max_length=512).to(model.device)
    
    summary_ids = model.generate(
        inputs["input_ids"],
        max_new_tokens=60,
        min_length=10,
        num_beams=4,
        no_repeat_ngram_size=3,        
        repetition_penalty=1.2,        
        early_stopping=True
    )
    
    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

# Test User Inputs
test_notice = "CIRCULAR: Library books must be returned by September 30. Faculty approvals needed for extensions."

print("\n--- USER GENERATION TEST ---")
print("Student View:", generate_persona_summary(test_notice, role="student"))
print("Faculty View:", generate_persona_summary(test_notice, role="faculty"))


--- USER GENERATION TEST ---
Student View: Summarize for student: CIRCULAR: Library books must be returned by September 30. Faculty approvals needed for extensions.
Faculty View: Summarize for faculty: CIRCULAR: Library books must be returned by September 30. Faculty approvals needed for extensions.


In [10]:
import evaluate
import spacy

# 1. ROUGE Metrics Evaluation
rouge = evaluate.load("rouge")

# Evaluate on the held-out eval split (not the training data) so ROUGE reflects
# generalization rather than memorization.
def split_role_and_text(input_text):
    role, _, notice = input_text.partition(": ")
    role = role.replace("summarize for ", "")
    return role, notice

predictions = []
references = []
for row in eval_dataset:
    role, notice = split_role_and_text(row["input_text"])
    predictions.append(generate_persona_summary(notice, role=role))
    references.append(row["target_summary"])

rouge_results = rouge.compute(predictions=predictions, references=references)
print("\n--- EVALUATION RESULTS ---")
print("ROUGE Metrics:", rouge_results)

# 2. Entity Hallucination Check
nlp = spacy.load("en_core_web_sm")

def check_date_hallucinations(source_text: str, generated_summary: str) -> bool:
    source_doc = nlp(source_text)
    summary_doc = nlp(generated_summary)
    
    source_dates = {ent.text.lower() for ent in source_doc.ents if ent.label_ in ["DATE", "TIME"]}
    summary_dates = {ent.text.lower() for ent in summary_doc.ents if ent.label_ in ["DATE", "TIME"]}
    
    # Check if generated summary contains dates missing from the source text
    hallucinated_dates = summary_dates - source_dates
    has_hallucination = len(hallucinated_dates) > 0
    
    if has_hallucination:
        print(f"Hallucination Warning! Generated ungrounded dates: {hallucinated_dates}")
    else:
        print("Date Check Passed: Zero date hallucinations detected.")
        
    return has_hallucination

# Test Guardrail Check
check_date_hallucinations(test_notice, generate_persona_summary(test_notice, role="student"))


--- EVALUATION RESULTS ---
ROUGE Metrics: {'rouge1': np.float64(0.3075500163571233), 'rouge2': np.float64(0.19617604617604617), 'rougeL': np.float64(0.28149091200432047), 'rougeLsum': np.float64(0.28149091200432047)}
Date Check Passed: Zero date hallucinations detected.


False